# Smart Irrigation - Professional Pipeline (v2: Pro Edition)
## Single Model Random Forest (Double Mission) avec Log-Volume

This notebook adopts the **Microsoft Team Data Science Process (TDSP)** lifecycle to structure the machine learning workflow from data acquisition to deployment.

In [ ]:
!pip install lime -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import random
from typing import Dict, Any
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import f1_score, mean_squared_error, classification_report, r2_score, confusion_matrix

import warnings
warnings.filterwarnings('ignore')
plt.style.use('ggplot')
shap.initjs()

print("✅ Pipeline Tools & Visualization Libraries Loaded.")

## TDSP Stage 1: Business Understanding & Environment Setup

In [ ]:
# --- GLOBAL CONFIGURATION ---
DATASET_PATH = r'C:\Users\bouda\Desktop\123\Smart_Irrigation_dataset.csv'
RANDOM_SEED = 42
THRESHOLD_DEFICIT = 35.0

# Enforce reproducibility
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print("✅ Global configuration and seeds applied.")

## TDSP Stage 2: Data Acquisition & Understanding (EDA & Feature Engineering)

In [ ]:
df = pd.read_csv(DATASET_PATH)

if 'crop_name' in df.columns: df = df.drop(columns=['crop_name'])
if 'soil_type' in df.columns:
    le_soil = LabelEncoder()
    df['soil_type'] = le_soil.fit_transform(df['soil_type'])

# Outlier handling
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in ['irrigate', 'irrigation_amount_m3']]
for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = np.clip(df[col], Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

# Agronomic Features
df['total_water_input'] = df['effective_rainfall_mm']
df['ETc'] = df['reference_evapotranspiration_ET0_mm_day'] * df['crop_coefficient_Kc']
df['water_storage_capacity'] = df['root_zone_depth_m'] * df['available_water_content_mm_per_m']
df['water_stress_index'] = (df['temperature_C'] * df['solar_radiation_MJ_m2_day']) / (df['humidity_%'] + 1)
df['evaporative_demand_proxy'] = (df['temperature_C'] + df['wind_speed_m_s']) * (100 - df['humidity_%']) / 100

# --- DATA VALIDATION ---
assert df.isnull().sum().sum() == 0, 'Data contains missing values!'
assert len(df) > 0, 'Dataset is empty after processing.'
print("✅ Features Engineered & Data Integrity Validated.\n")

# ==========================================
# 📊 VISUALISATION 1: TARGET DISTRIBUTION 
# ==========================================
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.countplot(x='irrigate', data=df, palette='viridis')
plt.title('Target Distribution: Irrigate (0 vs 1)')
plt.xlabel('Irrigation Decision (0 = No, 1 = Yes)')

plt.subplot(1, 2, 2)
sns.histplot(df[df['irrigation_amount_m3'] > 0]['irrigation_amount_m3'], bins=30, kde=True, color='blue')
plt.title('Volume Distribution (When Irrigate=1)')
plt.xlabel('Irrigation Amount (m3)')
plt.tight_layout()
plt.show()

# ==========================================
# 📊 VISUALISATION 2: RELATIONSHIP ANALYSIS
# ==========================================
plt.figure(figsize=(16, 12))
corr_matrix = df.corr()
# Isoler les corrélations fortes pour une meilleure lisibilité
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', mask=mask, linewidths=0.5)
plt.title('Relationship Analysis: Global Feature Correlation Heatmap', fontsize=16)
plt.show()

## TDSP Stage 3: Modeling & Evaluation (Single Random Forest Architecture)

In [ ]:
# EXCLUDE only the targets and raw rainfall
EXCLUDE = ['irrigate', 'irrigation_amount_m3', 'rainfall_mm']
FEATURES = [c for c in df.columns if c not in EXCLUDE]
X = df[FEATURES]
y = df[['irrigate', 'irrigation_amount_m3']]

print(f"Training Single Random Forest (Multi-Output) with {len(FEATURES)} features...")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y['irrigate'], random_state=RANDOM_SEED)

scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURES)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=FEATURES)

y_train_log = y_train.copy()
y_train_log['irrigation_amount_m3'] = np.log1p(y_train['irrigation_amount_m3'])

# --- MODEL TRAINING ---
model = RandomForestRegressor(n_estimators=150, max_depth=12, random_state=RANDOM_SEED)
model.fit(X_train_s, y_train_log)

# --- PREDICTION SINGLE MODEL (Double Mission) ---
y_pred = model.predict(X_test_s)
irr_decision_pred = (y_pred[:, 0] > 0.5).astype(int) # Classification
y_pred_vol = np.expm1(y_pred[:, 1]) # Regression

# --- EVALUATION GLOBALE (Totalité du Dataset) ---
f1 = f1_score(y_test['irrigate'], irr_decision_pred)
rmse = np.sqrt(mean_squared_error(y_test['irrigation_amount_m3'], y_pred_vol))
r2 = r2_score(y_test['irrigation_amount_m3'], y_pred_vol)

metrics_df = pd.DataFrame({
    'Metric': ['Global Target-1 F1-Score', 'Global Target-2 RMSE (m3)', 'Global Target-2 R² Score'],
    'Value': [f1, rmse, r2]
})
display(metrics_df.style.format({'Value': '{:.4f}'}).set_caption('Random Forest Evaluation Summary'))

# ==========================================
# 📊 VISUALISATION 3: CONFUSION MATRIX
# ==========================================
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test['irrigate'], irr_decision_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', annot_kws={"size": 14}, 
            xticklabels=['Predicted 0 (No)', 'Predicted 1 (Yes)'], 
            yticklabels=['Actual 0 (No)', 'Actual 1 (Yes)'])
plt.title('Confusion Matrix - Irrigation Classifier', fontsize=14)
plt.show()

# ==========================================
# 📊 VISUALISATION 4: NATIVE FEATURE IMPORTANCE
# ==========================================
feature_importances = model.feature_importances_
indices = np.argsort(feature_importances)[::-1][:15] # Top 15 features

plt.figure(figsize=(12, 6))
sns.barplot(x=feature_importances[indices], y=np.array(FEATURES)[indices], palette='mako')
plt.title('Native Feature Importance (Gini - Before SHAP)', fontsize=14)
plt.xlabel('Relative Importance (Tree structure)')
plt.show()

## TDSP Stage 3 (Cont.): Modeling Explainability (SHAP & LIME)
Comprendre précisément pourquoi l'Intelligence Artificielle prend ses décisions.

In [ ]:
# ==========================================
# 🔬 VISUALISATION 5A: SHAP ANALYSIS
# ==========================================
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test_s)
print("🌟 Explication Globale SHAP : L'impact mathématique de chaque trait sur la décision Globale")
shap.plots.bar(shap_values[:, :, 0], max_display=18)

# ==========================================
# 🔬 VISUALISATION 5B: LIME ANALYSIS
# ==========================================
try:
    import lime
    import lime.lime_tabular
    print("\n🌟 Explication Locale LIME : Analyse d'un scénario précis du Test Set")
    
    # Wrap pour forcer LIME à ignorer la sortie 'Volume' du Random Forest (Multi-Output) et se concentrer sur 'Irrigate'
    def custom_predict_for_lime(x_data):
        preds = model.predict(x_data)
        # Retourne juste la prédiction de décision (la colonne 0)
        return preds[:, 0]

    explainer_lime = lime.lime_tabular.LimeTabularExplainer(
        training_data=np.array(X_train_s),
        feature_names=FEATURES,
        class_names=['Decision Score'],
        mode='regression', # LIME handles single array outputs using regression mode
        random_state=RANDOM_SEED
    )
    
    # Explication de la décision sur la TOUTE PREMIÈRE ligne du jeu de test
    sample_choisi = X_test_s.iloc[0].values
    exp = explainer_lime.explain_instance(sample_choisi, custom_predict_for_lime, num_features=8)
    exp.show_in_notebook(show_table=True)
    
except ImportError:
    print("⚠️ La bibliothèque LIME n'est pas installée. Rendu LIME ignoré.")
except Exception as e:
    print(f"⚠️ Erreur LIME: {e}")

## TDSP Stage 5: Customer Acceptance (Simulation & Expert System Checks)

In [ ]:
def final_smart_irrigation_output(raw_row: pd.Series, threshold_deficit: float = THRESHOLD_DEFICIT) -> Dict[str, str]:
    """
    Applies Data Validation, Random Forest Multi-Output model, and physiological safety guards.
    """
    for f in FEATURES:
        if f not in raw_row.index: 
            raw_row[f] = 0.0
            print(f"⚠️ Missing feature '{f}' defaulted to 0.0")
    
    # 🛑 1. DATA VALIDATION GUARDS (Types & Limites rigoureuses)
    if raw_row['soil_type'] not in [0, 1, 2, 3]:
        return {
            "Decision": "ERROR ❌", 
            "Volume": "N/A", 
            "Raison": f"DATA ERROR: soil_type doit être 0, 1, 2 ou 3 (reçu: {raw_row['soil_type']})."
        }
        
    for pct_col in ['humidity_%', 'soil_moisture_%', 'field_capacity_%', 'wilting_point_%']:
        if not (0 <= raw_row[pct_col] <= 100):
            return {
                "Decision": "ERROR ❌", 
                "Volume": "N/A", 
                "Raison": f"DATA ERROR: Le pourcentage de {pct_col} ({raw_row[pct_col]}) est physiquement impossible."
            }
            
    if raw_row['wilting_point_%'] >= raw_row['field_capacity_%']:
        return {
            "Decision": "ERROR ❌", 
            "Volume": "N/A", 
            "Raison": "DATA ERROR: Le point de flétrissement ne peut pas être supérieur ou égal à la capacité au champ."
        }

    # 🛑 2. SENSOR ANOMALY GUARD (Conflit Pluie/Senseur Sol)
    if raw_row['effective_rainfall_mm'] > 10.0 and raw_row['soil_moisture_%'] <= (raw_row['wilting_point_%'] + 2.0):
        return {
            "Decision": "ALERT 🚨", 
            "Volume": "0.00 m3", 
            "Raison": "SENSOR ANOMALY: Impossible Physics! Heavy rain detected but soil is dry. Check soil sensor."
        }
    
    # PREPARATION SCALING
    row_df = pd.DataFrame([raw_row[FEATURES]], columns=FEATURES)
    row_s = scaler.transform(row_df)
    
    # 🛑 3. WATER DEFICIT GUARD (Output Logic Check)
    physical_deficit = raw_row['ETc'] - raw_row['total_water_input']
    is_guard_triggered = physical_deficit > threshold_deficit
    
    # PREDICTION RF MODEL UNIQUE (DOUBLE MISSION)
    pred_out = model.predict(row_s)[0]
    is_irrigate_pred = 1 if pred_out[0] > 0.5 else 0
    model_vol = np.expm1(pred_out[1])
    
    if is_guard_triggered:
        is_irrigate = 1
        amount = max(model_vol, physical_deficit)
        reason = "SAFETY RULE: Critical Water Deficit"
    else:
        is_irrigate = is_irrigate_pred
        amount = model_vol if is_irrigate == 1 else 0.0
        reason = "Random Forest Logic (IA Double Mission)"
    
    return {
        "Decision": "IRRIGATE 💧" if is_irrigate == 1 else "OK ✅", 
        "Volume": f"{amount:.2f} m3", 
        "Raison": reason
    }

def predict_live_scenario(sensor_data: Dict[str, float]) -> Dict[str, str]:
    try:
        df_live = pd.DataFrame([sensor_data])
        # Feature engineering calculé à partir du dictionnaire test
        df_live['total_water_input'] = df_live.get('effective_rainfall_mm', 0.0)
        df_live['ETc'] = df_live.get('reference_evapotranspiration_ET0_mm_day', 0.0) * df_live.get('crop_coefficient_Kc', 0.0)
        df_live['water_storage_capacity'] = df_live.get('root_zone_depth_m', 0.0) * df_live.get('available_water_content_mm_per_m', 0.0)
        df_live['water_stress_index'] = (df_live.get('temperature_C', 0.0) * df_live.get('solar_radiation_MJ_m2_day', 0.0)) / (df_live.get('humidity_%', 0.0) + 1)
        df_live['evaporative_demand_proxy'] = (df_live.get('temperature_C', 0.0) + df_live.get('wind_speed_m_s', 0.0)) * (100 - df_live.get('humidity_%', 0.0)) / 100
        
        return final_smart_irrigation_output(df_live.iloc[0])
    except Exception as e:
        return {"Error": f"Inference Failed: {str(e)}"}

# ==========================================================================
# --- Étape 6 : L'Examen Final (Le Test de terrain) ---
# On vérifie que l'IA ne se trompe pas. Si elle prédit d'arroser 
# alors qu'il vient de pleuvoir dans les données de test, on ajuste l'entrée.
# ==========================================================================
print("\n" + "="*50)
print("🚀 ÉTAPE 6 : L'EXAMEN FINAL (TEST DE L'IA SUR LE TERRAIN)")
print("="*50)

# Test 1 : Jour Aléatoire (Ligne #6 de notre DataSet : irrigate=1 car déficit élevé)
scenario_jour_aleatoire = {
    'soil_type': 3, 'crop_age_days': 37, 'day_of_year': 75,
    'temperature_C': 36.59, 'humidity_%': 93.87, 'effective_rainfall_mm': 0.35, # Presque pas de pluie
    'solar_radiation_MJ_m2_day': 7.68, 'wind_speed_m_s': 3.57,
    'field_capacity_%': 28.19, 'wilting_point_%': 16.94, 'soil_moisture_%': 21.0, # Humidité moyenne
    'reference_evapotranspiration_ET0_mm_day': 2.745, 'crop_coefficient_Kc': 0.915,
    'application_efficiency_%': 61.72, 'root_zone_depth_m': 1.135,
    'available_water_content_mm_per_m': 124.15, 'irrigation_interval_days': 10,
    'p_fraction': 0.405
}

# Test 2 : Chute de Pluie (Le même jour, soudainement il pleut 45mm ! Le sol se gorge d'eau)
scenario_chute_de_pluie = scenario_jour_aleatoire.copy()
scenario_chute_de_pluie['effective_rainfall_mm'] = 45.0  # Orage ! 
scenario_chute_de_pluie['soil_moisture_%'] = 28.0        # Le sol est rempli d'eau suite à l'orage

# Test 3 : Erreur de donnée utilisateur (Exemple visé : soil_type n'existe pas ou % absurde)
scenario_erreur_frappe = scenario_jour_aleatoire.copy()
scenario_erreur_frappe['soil_type'] = 12  # Le type 12 n'existe pas ! (Seul 0,1,2,3 sont valides)

# Exécution de l'Examen Final
tests = [
    ("☀️ TEST A : JOUR ALÉATOIRE SANS PLUIE (L'IA doit irriguer)", scenario_jour_aleatoire),
    ("🌧️ TEST B : CHUTE DE PLUIE NORMALE (L'IA doit comprendre qu'il ne faut PAS irriguer)", scenario_chute_de_pluie),
    ("❌ TEST C : ERREUR DE SAISIE CAPTEUR (Le soil_type vaut 12 - Le code doit lever une Erreur)", scenario_erreur_frappe)
]

for nom_test, data in tests:
    print(f"\n{nom_test}")
    res = predict_live_scenario(data)
    display(pd.DataFrame([res]))

## TDSP Stage 4: Deployment (ONNX Mobile Export & API Preparation)
Exporting the Random Forest for production APIs.

In [ ]:
import joblib
import os

# 1. Save the Scaler and Random Forest Model (Python Backend)
joblib.dump(scaler, 'feature_scaler.pkl')
joblib.dump(model, 'irrigation_rf_multi_model.pkl')
print("✅ Python Backend Models Saved (Single Multi-Output Model)")

# 2. Export to ONNX (Mobile App Target)
try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    initial_type = [('float_input', FloatTensorType([None, len(FEATURES)]))]
    
    onx_model = convert_sklearn(model, initial_types=initial_type)
    with open("irrigation_rf_model.onnx", "wb") as f:
        f.write(onx_model.SerializeToString())
        
    print("✅ Mobile Models Saved (irrigation_rf_model.onnx)")

except ImportError:
    print("⚠️ 'skl2onnx' not installed.")
except Exception as e:
    print(f"⚠️ Erreur lors de l'exportation ONNX : {e}")